# VectorDrugMatch - comparison against published-baseline methods (GPU)

Trains the four standard knowledge-graph-embedding baselines (**TransE, DistMult, ComplEx, RotatE**) - the model class the drug-repurposing
literature, including the TxGNN paper, benchmarks against - on **exactly the same** splits, leakage-safe graph and candidate pairs as
VectorDrugMatch, and measures VectorDrugMatch under the **zero-shot-disease protocol** that published methods use.

This is a **post-hoc comparison**: the deployed model was selected and frozen, and the holdout evaluated once, before any of this. Nothing
here changes the deployed model, and no model is selected on these numbers.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**.
2. Have **`vdm_primekg_for_colab.zip`** (4.8 MB, already in your Downloads folder) ready - Step 2 asks you to upload it.
   No Google Drive is used, so each stage downloads its own results zip to your browser when it finishes.

Code pinned to `6c20903a63591a88838bcec4f6c84a8afba93899`. Run the cells top to bottom. Every stage is resumable within a session (`--resume` skips finished runs); if the
runtime disconnects, re-run Steps 1-2 and then the stage you were on.

## Step 1 - GPU and code

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU, then run again'
print(torch.cuda.get_device_name(0))
%cd /content
!rm -rf VectorDrugMatch && git clone -q -b exp/final-audit-20261117 https://github.com/MUKUL-PRASAD-SIGH/VectorDrugMatch
%cd /content/VectorDrugMatch
!git checkout -q 6c20903a63591a88838bcec4f6c84a8afba93899 && git log -1 --oneline
%pip install -q torch-geometric

## Step 2 - Upload the data, verify every file by SHA-256
A file picker appears: choose **`vdm_primekg_for_colab.zip`** from your Downloads. The cell stops with an error if any file's hash differs
from the committed manifest. It must end with `DATA AND ALL SPLIT FILES VERIFIED`.

In [ ]:
import hashlib, json, shutil, os, zipfile
from google.colab import files
up = files.upload()
zipfile.ZipFile(list(up)[0]).extractall('data/processed')
man = json.load(open('Docs/release/vdm_v1_candidate_manifest.json'))['files']
want = {f['path']: f['sha256'] for f in man if f['path'].startswith('data/processed/primekg/') and f['path'].endswith(('.csv','.npy','.npz'))}
bad = [p for p, h in want.items() if hashlib.sha256(open(p,'rb').read()).hexdigest() != h]
assert not bad, f'DATA MISMATCH {bad}'
P = 'data/processed/primekg/splits_final_20261117'
shutil.rmtree(P, ignore_errors=True)
shutil.copytree('reports/final_audit_20261117/protocol_files', P)
ref = json.load(open('reports/final_audit_20261117/protocol_manifest_committed.json'))
ok = all(hashlib.sha256(open(f'{P}/{k}','rb').read()).hexdigest() == v['sha256'] for k, v in ref['files'].items())
ok &= hashlib.sha256(open(f'{P}/holdout_LOCKED.npz','rb').read()).hexdigest() == ref['holdout']['sha256']
Q = 'data/processed/primekg/splits_disease_20261118'
shutil.rmtree(Q, ignore_errors=True)
shutil.copytree('reports/final_audit_20261117/disease_protocol_files', Q)
ok &= hashlib.sha256(open(f'{Q}/disease_split.npz','rb').read()).hexdigest() == json.load(open(f'{Q}/manifest.json'))['file_sha256']
assert ok, 'SPLIT FILE MISMATCH - STOP'
os.makedirs('outputs/final_audit_20261117', exist_ok=True)

def save(tag):
    """Zip the results so far (without model weights) and download them to the browser."""
    z = f'/content/comparison_{tag}.zip'
    os.system(f"cd outputs && rm -f {z} && zip -rq {z} final_audit_20261117 -x '*/model.pt' -x '*/model.pth'")
    print(tag, 'zip MB:', round(os.path.getsize(z)/1e6, 2))
    files.download(z)

print('DATA AND ALL SPLIT FILES VERIFIED')

## Stage 1 (about 25 min) - the key comparison: KGE baselines on the locked holdout
Trained on `final_train`, the same training data as the deployed model `vdm-1.0.0`, so these numbers are directly comparable with its
headline result. Downloads `comparison_stage1.zip` when it finishes.

In [ ]:
!python -u scripts/final_audit/train_kge.py --models TransE DistMult ComplEx RotatE --splits final_train --seeds 42 --steps 12000 --eval-every 1000 --patience 4 --eval-holdout --resume --out outputs/final_audit_20261117/kge 2>&1 | grep -E 'DONE|early stop|FAIL'
save('stage1')

## Stage 2 (about 20 min) - the zero-shot-DISEASE protocol (the published setting)
Diseases are the held-out unit here (TxGNN's `complex_disease` setting), so these numbers are **not** comparable with the drug-disjoint
results and must never be mixed with them. Runs the VectorDrugMatch GCN (3 seeds), the feature baselines and the KGE baselines on that
split. Downloads `comparison_stage2.zip` when it finishes.

In [ ]:
!python -u scripts/final_audit/train_gcn_protocol.py --name L3_combo --splits disease_split --seeds 21 42 52 --layers 4 --emb 128 --head 64 --dropout 0.4 --wd 0.0005 --epochs 200 --patience 20 --resume --out outputs/final_audit_20261117/disease_protocol 2>&1 | grep -E 'DONE|early stop|FAIL'
!python -u scripts/final_audit/dev_cv_baselines.py --splits disease_split --skip-rf --out outputs/final_audit_20261117/disease_protocol 2>&1 | grep -E '^disease_split|Error'
!python -u scripts/final_audit/train_kge.py --models TransE DistMult ComplEx RotatE --splits disease_split --seeds 42 --steps 12000 --eval-every 1000 --patience 4 --resume --out outputs/final_audit_20261117/kge 2>&1 | grep -E 'DONE|early stop|FAIL'
save('stage2')

## Stage 3 (about 60 min, optional) - KGE baselines on the 5 development folds
Fold-to-fold variability on the same folds and candidate pairs as every VectorDrugMatch development number. The least critical stage: skip
it if you are short on time. Downloads `comparison_stage3.zip` when it finishes.

In [ ]:
!python -u scripts/final_audit/train_kge.py --models TransE DistMult ComplEx RotatE --splits cv_fold0 cv_fold1 cv_fold2 cv_fold3 cv_fold4 --seeds 42 --steps 12000 --eval-every 1000 --patience 4 --resume --out outputs/final_audit_20261117/kge 2>&1 | grep -E 'DONE|early stop|FAIL'
save('stage3')

## Step 4 - final check and download
Confirms every run used the GPU and every leakage audit passed, then downloads the complete `comparison_final.zip`. Send me that file.

In [ ]:
import json, glob, collections
rc = [json.load(open(f)) for f in glob.glob('outputs/final_audit_20261117/**/run_config.json', recursive=True)]
print(len(rc), 'runs |', dict(collections.Counter(str(r.get('gpu')) for r in rc)))
assert all(r.get('gpu') for r in rc), 'some runs did not use the GPU'
au = [json.load(open(f))['all_pass'] for f in glob.glob('outputs/final_audit_20261117/**/leakage_audit.json', recursive=True)]
print('leakage audits:', len(au), '| all pass:', all(au))
assert all(au), 'a leakage audit failed - STOP and report this'
save('final')